In [ ]:
import pandas as pd
import os
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import glob
import plotnine as gg
import scipy.stats as stats
import json
from essential.utils import get_hash, compute_topk_precision_metrics
from essential.ode import ODEstimator
from essential.utils import load_regulondb_full


ref_db = load_regulondb_full(drop_duplicates=True)

In [ ]:
BASE_DIR = "/workspace/experiments/11172025_presentation"
RESULTS_DIR = os.path.join(BASE_DIR, "runs_parametersweep")
FIGURE_DIR = os.path.join(BASE_DIR, "figures_parametersweep")
os.makedirs(FIGURE_DIR, exist_ok=True)

In [ ]:
adata = sc.read_h5ad("/workspace/data/250516_TF_perturbseq/250516_TF_perturbseq.annotated.h5ad")
targetted_tfs = adata.obs["consensus_target"].unique()
targetted_tfs = [t.lower() for t in targetted_tfs if t in adata.var_names]

In [ ]:
# directories = glob.glob("/workspace/results/250516_TF_perturbseq/ode_experiment_10282025/*")
directories = glob.glob(f"{RESULTS_DIR}/*")
all_dfs = []
tags_to_path = {}
for directory in directories:
    try:
        with open(os.path.join(directory, "config.json"), "r") as f:
            config = json.load(f)

        topk_df = pd.read_csv(os.path.join(directory, "topk_precision.csv"))
        all_dfs.append(topk_df)
        tag = config["tag"]
        tags_to_path[tag] = directory
        print(directory, tag)
    except Exception as e:
        print(f"Ignoring {directory}; missing files")
all_dfs = pd.concat(all_dfs)

In [ ]:
directories = glob.glob(f"{RESULTS_DIR}/*")
all_topk_history_dfs = []
for directory in directories:
    try:
        with open(os.path.join(directory, "config.json"), "r") as f:
            config = json.load(f)
        tag = config["tag"]
        print(directory, tag)
        topk_history_df = pd.read_csv(os.path.join(directory, "topk_history.csv")).assign(tag=tag)
        all_topk_history_dfs.append(topk_history_df)
    except Exception as e:
        print(f"Ignoring {directory}; missing files")
all_topk_history_dfs = pd.concat(all_topk_history_dfs)

In [ ]:
all_topk_history_dfs.query("tag == 'tanhdynamic_concentration_fixed'")

In [ ]:
plot_df = all_dfs.query("type == 'offdiag'")

plot_df.query("is_evidence == True").groupby("model").apply(lambda x: x.shape[0]).sort_values(
    ascending=False
)

In [ ]:
plot_df["model"].value_counts()

In [ ]:
models_to_keep = {
    "tanh_concentration_fixed_t": "cellbox (all batches, t)",
    # "tanh_concentration_fixed_onebatch_t": "cellbox (one batch, t)",
    "marginal_all": "marginal baseline",
}

In [ ]:
plot_df = all_dfs.query("type == 'offdiag'")
plot_df = plot_df.query("model in @models_to_keep")
plot_df["model"] = plot_df["model"].map(models_to_keep)

fig = (
    gg.ggplot(plot_df, gg.aes(x="topk", y="n_tp", color="model"))
    + gg.geom_line()
    + gg.theme_minimal()
    + gg.theme(figure_size=(6.5, 4))
    + gg.labs(x="top K predicted interactions", y="# of hits in RegulonDB")
)
fig.save(os.path.join(FIGURE_DIR, "precision_vs_topk_main.png"), dpi=300)
fig

In [ ]:
models_to_keep = {
    "tanh_concentration_fixed_t": "cellbox (all batches, t)",
    "tanh_concentration_fixed_onebatch_t": "cellbox (one batch, t)",
    # "marginal_all": "marginal baseline",
}

plot_df = all_dfs.query("type == 'offdiag'")
plot_df = plot_df.query("model in @models_to_keep")
plot_df["model"] = plot_df["model"].map(models_to_keep)

fig = (
    gg.ggplot(plot_df, gg.aes(x="topk", y="n_tp", linetype="model"))
    # + gg.scale_color_manual(values=models_to_keep_colors)
    + gg.geom_line()
    + gg.theme_minimal()
    + gg.theme(figure_size=(6.5, 4))
    + gg.labs(x="top K predicted interactions", y="# of hits in RegulonDB")
)
fig.save(os.path.join(FIGURE_DIR, "precision_vs_topk_batch.png"), dpi=300)
fig

In [ ]:
models_to_keep = {
    "tanh_concentration_fixed_t": "cellbox (all batches, t)",
    "tanhdynamic_concentration_fixed_onebatch_t": "cellbox (one batch, t)",
    # "marginal_all": "marginal baseline",
}

plot_df = all_dfs.query("type == 'offdiag'")
plot_df = plot_df.query("model in @models_to_keep")
plot_df["model"] = plot_df["model"].map(models_to_keep)

fig = (
    gg.ggplot(plot_df, gg.aes(x="topk", y="n_tp", linetype="model"))
    # + gg.scale_color_manual(values=models_to_keep_colors)
    + gg.geom_line()
    + gg.theme_minimal()
    + gg.theme(figure_size=(6.5, 4))
    + gg.labs(x="top K predicted interactions", y="# of hits in RegulonDB")
)
fig.save(os.path.join(FIGURE_DIR, "precision_vs_topk_batch.png"), dpi=300)
fig

# Venn

In [ ]:
plot_df.loc[:, "regulatory_pair"] = (
    plot_df.loc[:, "target_gene"] + "_" + plot_df.loc[:, "regulator_gene"]
)

best_ours = plot_df.loc[lambda x: x["model"] == "cellbox (all batches, t)"]
best_baseline = plot_df.loc[lambda x: x["model"] == "marginal baseline"]

print(best_ours.shape[0], best_baseline.shape[0])

In [ ]:
discoveries_ours = best_ours.loc[:, "regulatory_pair"].values
discoveries_marginal = best_baseline.loc[:, "regulatory_pair"].values

# Overall performance

from matplotlib_venn import venn2

set1 = set(discoveries_ours)
set2 = set(discoveries_marginal)

venn2([set1, set2], ("cellbox", "marginal"))

plt.savefig(os.path.join(FIGURE_DIR, "venn_top3k.png"), dpi=300)
plt.show()

In [ ]:
discoveries_ours = best_ours.query("is_evidence == True").loc[:, "regulatory_pair"].values
discoveries_marginal = best_baseline.query("is_evidence == True").loc[:, "regulatory_pair"].values

# Overall performance

from matplotlib_venn import venn2

set1 = set(discoveries_ours)
set2 = set(discoveries_marginal)

venn2([set1, set2], ("cellbox", "marginal"))
plt.savefig(os.path.join(FIGURE_DIR, "venn_hits_in_regulondb.png"), dpi=300)
plt.show()

# individual model lookup

In [ ]:
directory = tags_to_path["tanh_concentration_fixed_lambda_0.1"]
print(directory)
# directory = tags_to_path["tanh_concentration_fixed"]
amat = np.load(os.path.join(directory, "Amat.npz"), allow_pickle=True)
amat_df = pd.DataFrame(amat["matrix"], index=amat["genes"], columns=amat["genes"])
step_df = pd.read_csv(os.path.join(directory, "step_history.csv"))
epoch_df = pd.read_csv(os.path.join(directory, "history.csv"))

In [ ]:
vals = amat_df.values.flatten()
abs_vals = np.abs(vals)

plt.hist(vals, bins=100)
plt.yscale("log")
plt.show()

In [ ]:
step_df

In [ ]:
plt.plot(step_df["l1_prior"], label="l1_prior")
plt.yscale("log")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 1, figsize=(10, 5))
plt.plot(step_df["reco_loss"], label="reco_loss")
plt.plot(step_df["l1_prior"], label="l1_prior")
plt.legend()
plt.yscale("log")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
epoch_df["train_reco_loss"].plot(ax=axes[0])
axes[0].set_yscale("log")
axes[0].set_title("train")
epoch_df["val_reco_loss"].plot(ax=axes[1])
axes[1].set_yscale("log")
axes[1].set_title("val")
plt.show()

In [ ]:
processed_a_mat = ODEstimator.process_interaction_matrix(amat_df, return_square=False, delta=0.1)
processed_a_mat_subset = processed_a_mat.loc[lambda x: x["target_gene"].isin(targetted_tfs)]

topk_precision_df = compute_topk_precision_metrics(processed_a_mat, "model")
topk_precision_subset_df = compute_topk_precision_metrics(processed_a_mat_subset, "model")

display(topk_precision_df.query("type == 'offdiag'").groupby("model")["is_evidence"].value_counts())
display(
    topk_precision_subset_df.query("type == 'offdiag'")
    .groupby("model")["is_evidence"]
    .value_counts()
)